# SOXX 近30日期权固定 25Δ 偏斜：来源可行性预检

本卡只读取已经冻结且已取得的四份 IBKR 市场数据快照，不调用券商、不拟合模型、不回放账户。目标是在同一 2026-11-20 到期日内核实 ±25Δ 两侧是否被固定行权价夹住，再按事先指定的 Delta 线性插值计算一次来源读数。此前三快照权限门出现并发重复，本卡使用独立排他执行声明和四次新快照预算。

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math
root=Path.cwd().resolve();card=root/'research/ibkr_soxx_25delta_source_preflight_20261022'
plan=json.loads((card/'plan.json').read_text());claim=json.loads((card/'execution_claim.json').read_text());observed=json.loads((card/'observations.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen_before_market_reads'] and claim['status']=='claimed_before_market_read'
assert observed['plan_sha256']==claim['plan_sha256']==sha(card/'plan.json')
assert sha(root/plan['parent_regular_hours_result'])==plan['parent_regular_hours_result_sha256']
assert sha(root/plan['fixed_chain'])==plan['fixed_chain_sha256']
assert set(observed['entries'])=={x['role'] for x in plan['contracts']}
assert all(observed[x]==0 for x in ('broker_order_calls','alerts','account_reads','model_fits','native_accounts'))
starts=[];ends=[];values={}
for row in plan['contracts']:
 role=row['role'];item=observed['entries'][role];assert item['strike']==row['strike'] and not item['tool_is_error']
 start=datetime.fromisoformat(item['requested_at_utc'].replace(' UTC','+00:00'))
 end=datetime.fromisoformat(item['received_at_utc'].replace(' UTC','+00:00'))
 assert start>=datetime.fromisoformat(plan['not_before_utc']) and end<=datetime.fromisoformat(plan['not_after_utc']) and end>=start
 starts.append(start);ends.append(end)
 assert json.loads(item['raw_response'])==item['parsed_response']
 quote=item['parsed_response'];bidask=quote['bid-ask'];iv=quote['option-midpoint-iv'];greeks=quote['option-greeks'];delta=greeks['delta']
 assert quote['top-status']['status']=='REALTIME'
 assert all(math.isfinite(float(bidask[key])) for key in ('bid','ask')) and 0<float(bidask['bid'])<=float(bidask['ask'])
 assert iv['isValid'] is True and math.isfinite(float(iv['annualIv'])) and iv['annualIv']>0
 assert math.isfinite(float(delta)) and (delta>0 if role.startswith('call') else delta<0)
 values[role]={'delta':float(delta),'annual_iv':float(iv['annualIv']),'bid':float(bidask['bid']),'ask':float(bidask['ask'])}
span=(max(ends)-min(starts)).total_seconds();assert span<=plan['max_request_span_seconds']
call_a,call_b=values['call_low_strike'],values['call_high_strike'];put_a,put_b=values['put_low_strike'],values['put_high_strike']
assert call_a['delta']>0.25>call_b['delta'] and put_a['delta']>-0.25>put_b['delta']
def linear_iv(target,a,b):
 return a['annual_iv']+(target-a['delta'])*(b['annual_iv']-a['annual_iv'])/(b['delta']-a['delta'])
call_iv=linear_iv(.25,call_a,call_b);put_iv=linear_iv(-.25,put_a,put_b)
result={'schema_version':1,'status':'fixed_delta25_brackets_observed_source_feasibility_only','plan_sha256':sha(card/'plan.json'),'observations_sha256':sha(card/'observations.json'),'claim_sha256_before_completion':sha(card/'execution_claim.json'),'request_span_seconds':span,'expiration':plan['expiration'],'exchange':plan['exchange'],'call_25delta_annual_iv':call_iv,'put_minus25delta_annual_iv':put_iv,'put_minus_call_annual_iv':put_iv-call_iv,'skew_volatility_points':100*(put_iv-call_iv),'quotes_realtime':True,'four_brackets_valid':True,'option_quote_timestamps_provided':False,'historical_2011_2022_coverage_verified':False,'historical_PIT_qualified':False,'qualified_forward_strategy_days':0,'model_fits':0,'native_accounts':0,'broker_order_calls':0,'claim_budget_used':4,'decision':'source_feasibility_only_no_Stage1_feature_or_promotion'}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({k:result[k] for k in ('status','request_span_seconds','call_25delta_annual_iv','put_minus25delta_annual_iv','skew_volatility_points','qualified_forward_strategy_days')})

{'status': 'fixed_delta25_brackets_observed_source_feasibility_only', 'request_span_seconds': 7.0, 'call_25delta_annual_iv': 0.39583034884422874, 'put_minus25delta_annual_iv': 0.4510082617855091, 'skew_volatility_points': 5.517791294128038, 'qualified_forward_strategy_days': 0}


## 中文最终结论

2026-10-22 常规时段、同一 2026-11-20 到期日的四个固定 SOXX 期权报价均为 REALTIME，IV 与 Delta 有效；530/540 看涨、450/460 看跌分别夹住 ±25Δ。七秒请求窗口内的线性插值给出看涨 25Δ IV 0.395830、看跌 −25Δ IV 0.451008，差值 **5.518 波动率百分点**。原始响应和冻结计划的五个对象已在 NAS 归档并逐一哈希读回。

决定：`source_feasibility_only_no_Stage1_feature_or_promotion`。本卡 0 拟合、0 账户、0 券商订单、0 合格前向策略日。连接器未给出期权报价自身时间戳，2011–2022 历史合约/IV/Delta 原始版本也缺失；这个单次读数不得进入 Stage1 或证明 SOXX 超额。下一步需独立冻结带排他声明的多日期来源捕获，并先向历史数据供应商核实 SOXX 全训练期覆盖与决策前可用性，不在本卡补选行权价或追加报价。